### In this notebook we show how the model to model evaluation module works

In [1]:
import sys, json

sys.path.append("../")
sys.path.append("../model_evaluation/")

In [23]:
# load some examples from examples folder in Signavio json format
filename_ground_truth = f"../examples/misc_booking_flight_tickets.json"
with open(filename_ground_truth, "r") as infile:
    model_1 = json.load(infile)


filename_generated = f"../examples/misc_booking_variant.json"
with open(filename_generated, "r") as infile:
    model_2 = json.load(infile)

In [24]:
import json
from BPMN_conversion import BPMNConverter


model_1_json = json.loads(BPMNConverter.convert(model_1).to_json())
# misc_bft_json_original = original_BPMNConverter.convert(E4_1).to_json()

model_2_json = json.loads(BPMNConverter.convert(model_2).to_json())

In [25]:
# ==============================================================================
# BPMN Model Comparison Pipeline
# ==============================================================================
import json
from rendering import create_similarity_dashboard, print_similarity_report
from bpmn_normalization import normalize_atomic_names
from bpmn_similarity import calculate_bpmn_similarity
from utils import cosine_sim_optimized


print("BPMN MODEL COMPARISON PIPELINE")


# Step 1: Model Summary
print("\n[1] MODEL STATISTICS")


def count_elements(model):
    """Count BPMN elements in a model."""
    return {
        "activities": len(model.get("activities", [])),
        "events": len(model.get("events", [])),
        "gateways": len(model.get("gateways", [])),
        "sequence_flows": len(model.get("sequenceFlows", [])),
        "message_flows": len(model.get("messageFlows", [])),
        "pools": len(model.get("pools", [])),
        "lanes": sum(len(p.get("lanes", [])) for p in model.get("pools", [])),
    }


model1_counts = count_elements(model_1_json)
model2_counts = count_elements(model_2_json)

print(f"Model 1: {sum(model1_counts.values())} total elements")
for key, val in model1_counts.items():
    if val > 0:
        print(f"  • {key.replace('_', ' ').title()}: {val}")

print(f"\nModel 2: {sum(model2_counts.values())} total elements")
for key, val in model2_counts.items():
    if val > 0:
        print(f"  • {key.replace('_', ' ').title()}: {val}")

# Step 2: Normalize Names
print("\n[2] SEMANTIC NAME NORMALIZATION")

threshold = 0.6
print(f"Aligning element names using a sentence transformer model (threshold={threshold})...")

model2_aligned, mappings = normalize_atomic_names(model_1_json, model_2_json, cosine_sim_optimized, threshold=threshold)

total_mappings = sum(len(v) for v in mappings.values())
if total_mappings > 0:
    print(f"✓ Applied {total_mappings} semantic name mappings")
    for elem_type, mapping in mappings.items():
        if mapping:
            print(f"  • {elem_type}: {len(mapping)} mappings")
            # Show first example
            first_old, first_new = next(iter(mapping.items()))
            print(f"    Example: '{first_old}' → '{first_new}'")
else:
    print("✓ No mappings needed (names already aligned)")

# Step 3: Calculate Similarity Without Normalization
print("\n[3] SIMILARITY ANALYSIS")


similarity_without_norm = calculate_bpmn_similarity(model_1_json, model_2_json, method="dice")

similarity_with_norm = calculate_bpmn_similarity(model_1_json, model2_aligned, method="dice")

print("WITHOUT normalization:")
print(f"  Overall Similarity: {similarity_without_norm['overall']:.1%}")
for cat in ["structural", "flows", "organizational", "subprocess"]:
    score = similarity_without_norm["high_level_scores"][cat]
    print(f"    • {cat.title()}: {score:.1%}")

print("\nWITH normalization:")
print(f"  Overall Similarity: {similarity_with_norm['overall']:.1%}")
for cat in ["structural", "flows", "organizational", "subprocess"]:
    score = similarity_with_norm["high_level_scores"][cat]
    print(f"    • {cat.title()}: {score:.1%}")

improvement = similarity_with_norm["overall"] - similarity_without_norm["overall"]
print(f"\n  → Improvement: {improvement:+.1%} ({abs(improvement)*100:.1f} percentage points)")

# Store results for dashboard
similarity_results = similarity_with_norm


print("Pipeline complete. Results stored in 'similarity_results'.")

BPMN MODEL COMPARISON PIPELINE

[1] MODEL STATISTICS
Model 1: 45 total elements
  • Activities: 4
  • Events: 11
  • Gateways: 3
  • Sequence Flows: 15
  • Message Flows: 7
  • Pools: 3
  • Lanes: 2

Model 2: 49 total elements
  • Activities: 4
  • Events: 12
  • Gateways: 4
  • Sequence Flows: 17
  • Message Flows: 7
  • Pools: 3
  • Lanes: 2

[2] SEMANTIC NAME NORMALIZATION
Aligning element names using a sentence transformer model (threshold=0.6)...
✓ Applied 31 semantic name mappings
  • activity_names: 4 mappings
    Example: 'Select best room and book' → 'Select the best offer and request tickets'
  • activity_types: 5 mappings
    Example: 'Task' → 'Task'
  • event_names: 10 mappings
    Example: 'Receive confirmation' → 'Receive confirmation'
  • event_types: 7 mappings
    Example: 'IntermediateMessageEventCatching' → 'IntermediateMessageEventCatching'
  • gateway_types: 2 mappings
    Example: 'Parallel' → 'Parallel'
  • pool_names: 3 mappings
    Example: 'Hotel Chain' → 'Tra

In [26]:
import json
from rendering.dashboard import create_similarity_dashboard


# Create and display dashboard
dashboard = create_similarity_dashboard(
    model_1_json,
    model_2_json,
    similarity_func=cosine_sim_optimized,
    calculate_similarity_func=calculate_bpmn_similarity,
    normalize_func=normalize_atomic_names,
    initial_threshold=0.5,
)
dashboard.display()

In [27]:
# Use the reusable XML embed function
import importlib
import rendering

importlib.reload(rendering)

# Load a BPMN XML file and render it
with open("../examples/linear_sequence.xml", "r", encoding="utf-8") as f:
    xml_str = f.read()

# Navigated viewer enables zoom/pan
rendering.render_bpmn_xml_embed(xml_str, height_px=500, navigated=True)

/Users/I585907/Develop/process-evaluation-framework/.venv/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


### Trace extraction

In [28]:
from json_to_pn import parse_simplified_bpmn_json

parse_simplified_bpmn_json(model_1_json)

({'sid-E5689480-97F3-457D-B623-EF8AA5626B99': ['sid-6CFCAB74-F99D-40F4-8D63-C630067F770C'],
  'sid-C30095C3-EE47-46D1-992F-3D1CBAEE726E': ['sid-29CC4EF1-285F-4448-9F16-9A59BC4D2961',
   'sid-656A3861-9BAE-474B-938D-FAD71080CF81'],
  'sid-BDBF2D9E-F414-4EB1-99E0-1E9C4D319DFF': ['sid-82D08B4D-EAFA-4493-8AEF-B033CA5330B4'],
  'sid-BD21A3DC-0E6D-46A2-A745-5F69C3991B8E': ['sid-A90FEC34-12FF-4446-B1CA-4296F231B6BC'],
  'sid-9D6D682F-6E36-4DDD-BED8-15A5A5EB3296': ['sid-B05C99F6-88AE-481F-866E-B477C4F03114'],
  'sid-6DE31EA7-8E36-459E-80CC-4E9B21E358B9': ['sid-5519292D-41B6-42EC-A1C1-59301B17A9B1',
   'sid-38BE7F50-8A2C-49F4-91C2-5B3BAB34D1A1'],
  'sid-C3683C28-0D9F-4D82-A8BA-EC504988B902': ['sid-057B0310-5D3D-43D5-AC58-401CCB61DFED'],
  'sid-08598060-42EA-4673-B1AC-33F41067FDC5': ['sid-1937D7C5-7A29-41CF-A357-01920694D23E'],
  'sid-7DBABD5C-7A71-4530-89FD-EB705ABDB000': [],
  'sid-4F7BA55F-A01B-4E6B-B083-49DDC83AB4F6': ['sid-90D1A4A6-96E9-4751-A514-3B89A25D029B'],
  'sid-223B9831-71E3-42E7-B4

In [ ]:
# Extract traces from both models
from trace_extraction import (
    extract_traces,
    compare_trace_sets,
    print_trace_comparison
)

print("\n[4] TRACE EXTRACTION")
print("Extracting execution traces (variants) from both models...")
print("This converts models to Petri nets and explores possible execution paths.\n")

# Extract traces with activity names (more readable)
traces_1 = extract_traces(
    model_1_json,
    timeout_seconds=2.0,
    max_loop_depth=3,
)

traces_2 = extract_traces(
    model2_aligned,  # Use normalized version
    timeout_seconds=2.0,
    max_loop_depth=3,
)

print(f"Model 1: Extracted {len(traces_1)} unique trace variants")
print(f"Model 2: Extracted {len(traces_2)} unique trace variants")

# Show sample traces from each model
print("\nSample traces from Model 1 (first 3):")
for i, trace in enumerate(list(traces_1)[:3], 1):
    print(f"  {i}. {' → '.join(trace)}")

print("\nSample traces from Model 2 (first 3):")
for i, trace in enumerate(list(traces_2)[:3], 1):
    print(f"  {i}. {' → '.join(trace)}")





[4] TRACE EXTRACTION
Extracting execution traces (variants) from both models...
This converts models to Petri nets and explores possible execution paths.

Model 1: Extracted 4 unique trace variants
Model 2: Extracted 2 unique trace variants

Sample traces from Model 1 (first 3):
  1. sid-044B6E47-536D-4262-B6A0-102D3C20A06A
  2. sid-65AC7D4E-E93A-41DF-942C-28A7D96D6275 → sid-4780FC45-2589-47CD-8E26-43E64C2D486D → sid-C30095C3-EE47-46D1-992F-3D1CBAEE726E → sid-4F7BA55F-A01B-4E6B-B083-49DDC83AB4F6 → sid-BDBF2D9E-F414-4EB1-99E0-1E9C4D319DFF → sid-D577E480-5715-4431-9E35-BBDA8806D8F6 → sid-C3683C28-0D9F-4D82-A8BA-EC504988B902 → sid-CD629142-A4C4-40EE-B7FF-EB0660A3ECAE → sid-7DBABD5C-7A71-4530-89FD-EB705ABDB000
  3. sid-619F6299-172A-4AFD-8C60-15267AF1D0C8

Sample traces from Model 2 (first 3):
  1. sid-28025B9D-3C9A-E519-4995-AC2C7E967137
  2. sid-7B7E3D4A-6965-1A96-4D82-902E81C834A8


In [30]:
# Compare trace sets comprehensively
comparison = compare_trace_sets(
    traces_1,
    traces_2,
    model_1_name="Model 1",
    model_2_name="Model 2 (normalized)"
)

# Print formatted comparison report
print_trace_comparison(comparison, show_traces=True)


TRACE COMPARISON: Model 1 vs Model 2 (normalized)

Model 1 Statistics:
  • Variants: 4
  • Trace length: 1-13 (avg: 6.0)
  • Unique activities: 16

Model 2 (normalized) Statistics:
  • Variants: 2
  • Trace length: 1-1 (avg: 1.0)
  • Unique activities: 2

Similarity Scores:
  • Jaccard: 0.00%
  • Dice: 0.00%
  • Overlap: 0.00%

Trace Coverage:
  • Common variants: 0
  • Only in Model 1: 4
  • Only in Model 2 (normalized): 2

Unique to Model 1 (first 3):
  1. sid-65AC7D4E-E93A-41DF-942C-28A7D96D6275 → sid-4780FC45-2589-47CD-8E26-43E64C2D486D → sid-C30095C3-EE47-46D1-992F-3D1CBAEE726E → sid-4F7BA55F-A01B-4E6B-B083-49DDC83AB4F6 → sid-BDBF2D9E-F414-4EB1-99E0-1E9C4D319DFF → sid-D577E480-5715-4431-9E35-BBDA8806D8F6 → sid-C3683C28-0D9F-4D82-A8BA-EC504988B902 → sid-CD629142-A4C4-40EE-B7FF-EB0660A3ECAE → sid-7DBABD5C-7A71-4530-89FD-EB705ABDB000
  2. sid-044B6E47-536D-4262-B6A0-102D3C20A06A
  3. sid-619F6299-172A-4AFD-8C60-15267AF1D0C8

Unique to Model 2 (normalized) (first 3):
  1. sid-28025B9

In [31]:
# Combine structural and trace similarity for final score
structural_sim = similarity_with_norm['overall']
trace_sim = comparison['jaccard_similarity']

# Weighted combination (adjust weights as needed)
structural_weight = 0.6
trace_weight = 0.4

combined_similarity = (structural_weight * structural_sim) + (trace_weight * trace_sim)

print(f"\n{'='*70}")
print("COMBINED SIMILARITY SCORE")
print(f"{'='*70}")
print(f"Structural Similarity:  {structural_sim:.2%} (weight: {structural_weight})")
print(f"Trace Similarity:       {trace_sim:.2%} (weight: {trace_weight})")
print(f"Combined Similarity:    {combined_similarity:.2%}")
print(f"{'='*70}\n")

print("✓ Complete evaluation pipeline finished!")
print("  Models compared using both structural and behavioral (trace) similarity.")


COMBINED SIMILARITY SCORE
Structural Similarity:  46.27% (weight: 0.6)
Trace Similarity:       0.00% (weight: 0.4)
Combined Similarity:    27.76%

✓ Complete evaluation pipeline finished!
  Models compared using both structural and behavioral (trace) similarity.
